# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a demonstration of loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset loaded: {getattr(metadata, 'name', None)}")
print(f"Description: {getattr(metadata, 'description', None)}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s from the Croissant schema.

In [ ]:
# List the available record sets and their fields (all with @id)
print("--- Available Record Sets (@id / name) ---")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"@id: {rs.id}")
    print(f"  name: {getattr(rs, 'name', None)}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - @id: {field.id} / name: {getattr(field, 'name', None)} / dataType: {getattr(field, 'data_type', None)}")
    print("  Columns:")
    for column in getattr(rs, 'columns', []):
        print(f"    - @id: {getattr(column, 'id', None)} / name: {getattr(column, 'name', None)}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
All record set, field, and column references use their `@id` as given in the previous overview.

In [ ]:
# For this dataset, we examine the main record set (often the one listing protein records).
# Let's select the first record set for demonstration—replace with correct @id as needed.
# Print available record set @ids again for reference:

main_record_set_id = record_sets[0].id if record_sets else None
print(f"Using record set: {main_record_set_id}")

# List all record sets @id for reference
record_set_ids = [rs.id for rs in record_sets]

dataframes = {}
for record_set in record_set_ids:
    records = list(dataset.records(record_set=record_set))
    dataframes[record_set] = pd.DataFrame(records)
    print(f"{record_set}: {dataframes[record_set].shape} shape")

# Show all columns (fields by @id) for the main record set
if main_record_set_id is not None:
    print(f"Columns for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalization, grouping. Replace `numeric_field_id` and `group_field_id` with the actual field `@id` values from the data overview above.

See code comments for where to fill in / customize for this dataset.

In [ ]:
# Example: Choose the field representing protein coverage or molecular weight, as these are numeric.
# Replace this with the correct field @id as seen in the record set overview (e.g., "coverage_percent", "MW", etc).
numeric_field_id = None
possible_numeric_fields = dataframes[main_record_set_id].select_dtypes('number').columns.tolist()
if possible_numeric_fields:
    # Taking first numeric column by default for demonstration
    numeric_field_id = possible_numeric_fields[0]
print(f"Using numeric field: {numeric_field_id}")

threshold = dataframes[main_record_set_id][numeric_field_id].quantile(0.9) if numeric_field_id else 10
filtered_df = dataframes[main_record_set_id]
if numeric_field_id is not None:
    filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head(5))
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a grouping field (e.g. "sample_id" or "accession") if relevant
group_field_id = None
for col in dataframes[main_record_set_id].columns:
    if 'sample' in col.lower() or 'accession' in col.lower():
        group_field_id = col
        break
if group_field_id:
    # Only try groupby if field available
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    display(grouped_df.head(5))

## 5. Visualization
Visualize distributions or relationships between fields. Here we plot a histogram and a boxplot of the selected numeric field for the record set.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set_id][numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Histogram of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    plt.figure(figsize=(5,4))
    sns.boxplot(y=dataframes[main_record_set_id][numeric_field_id].dropna())
    plt.title(f"Boxplot of {numeric_field_id}")
    plt.ylabel(numeric_field_id)
    plt.show()

if group_field_id and numeric_field_id:
    plt.figure(figsize=(16,6))
    sns.boxplot(x=dataframes[main_record_set_id][group_field_id], y=dataframes[main_record_set_id][numeric_field_id])
    plt.xticks(rotation=90)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step how to:
- Load protein mass spectrometry data from a FAIR^2 Croissant schema using only the `@id` fields.
- Explore record sets and fields by their stable identifiers.
- Extract and analyze records, normalize numeric fields, filter, and visualize.

Further exploration could include linking to external databases (e.g. UniProt), applying advanced statistics, or integrating with biological annotation tools.